#### Keras Tuner | Hyperparameter Tuning

Hyperparameter tuning manually (guessing learning rates and layer sizes) is exhausting. **Keras Tuner** automates this by systematically searching for the absolute perfect combination of hyperparameters for your specific dataset.

Instead of hardcoding a neural network, you build a **dynamic model builder function**. You give the Tuner a "Search Space" (e.g., *"try learning rates of 0.01, 0.001, or 0.0001"*), and it handles the training and tracking for you.

Here are the 3 main algorithms Keras Tuner uses:

1. **Random Search:** Just tries combinations entirely at random. A good baseline, but not very smart.

2. **Hyperband (Industry Standard):** Uses a "tournament" style approach. It trains a ton of different models for just 2 or 3 epochs, ruthlessly kills off the worst-performing ones, and only gives full training time to the top survivors. It is incredibly fast.

3. **Bayesian Optimization:** Uses advanced probability. It looks at the results of its past trials to mathematically guess which combination of parameters it should try next.

Here is the 4-step process it follows:

* **Step 1: The Model Builder:** You pass a special `hp` (HyperParameter) object into your function. Instead of writing `units=64`, you write `units=hp.Int('units', min_value=32, max_value=256, step=32)`.
* **Step 2: The Tuner:** You instantiate the tuner (e.g., `kt.Hyperband`), telling it to minimize Validation Loss.
* **Step 3: The Search:** You run `tuner.search()`. This looks exactly like a standard `model.fit()` loop.
* **Step 4: The Extraction:** You pull out the absolute best model and its perfect parameters.


In [1]:
!pip install keras_tuner

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tuner]
   -------------------- ------------------- 1/2 [keras_tune


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import tensorflow as tf
import keras_tuner as kt
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

C:\Users\Hp\AppData\Roaming\Python\Python311\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df = pd.read_csv('diabetes.csv')
X = df.drop(columns=['Outcome'],axis=1)
y = df[['Outcome']]
X,y

(     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
 0              6      148             72             35        0  33.6   
 1              1       85             66             29        0  26.6   
 2              8      183             64              0        0  23.3   
 3              1       89             66             23       94  28.1   
 4              0      137             40             35      168  43.1   
 ..           ...      ...            ...            ...      ...   ...   
 763           10      101             76             48      180  32.9   
 764            2      122             70             27        0  36.8   
 765            5      121             72             23      112  26.2   
 766            1      126             60              0        0  30.1   
 767            1       93             70             31        0  30.4   
 
      DiabetesPedigreeFunction  Age  
 0                       0.627   50  
 1                    

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
baseline_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)), # 8 features in diabetes data
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

C:\Users\Hp\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
baseline_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [7]:
baseline_model.fit(X_train_scaled, y_train, epochs=30, validation_split=0.2, verbose=0)

In [8]:
_, baseline_acc = baseline_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Baseline Test Accuracy: {baseline_acc:.4f}\n")

Baseline Test Accuracy: 0.7338



In [19]:
def build_model(hp):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(8,))) 
    
    for i in range(hp.Int('num_layers', 1, 3)):

        model.add(tf.keras.layers.Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=128, step=32),
            activation='relu'
        ))

        model.add(tf.keras.layers.Dropout(
            rate=hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)
        ))

    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

you need the `build_model` function because Keras Tuner isn't just training *one* model—it is building and destroying *hundreds* of different models behind the scenes.

### The Problem with Normal Keras Code

Normally, when you write `model = tf.keras.Sequential([...])`, you are building a **Static Model**. It is like pouring concrete. Once you define it has 32 neurons, it is permanently locked at 32 neurons.

If you hand that static model to a Tuner, it can't change the number of layers or neurons because the model is already built.

### The Solution: The "Factory" Function

By wrapping your code inside `def build_model(hp):`, you are no longer building a single model. You are creating a **Model Factory**.

Here is exactly what Keras Tuner does under the hood when you hit `tuner.search()`:

1. **Trial 1:** The Tuner creates an `hp` object with instructions *(e.g., "Make 1 layer with 32 neurons")*. It passes this `hp` into your `build_model(hp)` function. Your function builds that exact model, returns it, and the Tuner trains it.
2. **Trial 2:** The Tuner destroys the first model. It creates a brand new `hp` object with new instructions *(e.g., "Make 3 layers with 128 neurons")*. It calls your function *again*. Your function builds this new, massive model, returns it, and the Tuner trains it.
3. **Trial 3 to 100:** It repeats this process over and over.

In [10]:
tuner = kt.Hyperband(
    build_model,
    objective='val_loss',         # The referee watches Validation Loss
    max_epochs=20,                # Max epochs any single model gets to train
    factor=3,                     # Reduction factor for the tournament brackets
    directory='my_tuning_dir',    # Folder to save the trial logs
    project_name='customer_churn' # Sub-folder name
)

Reloading Tuner from my_tuning_dir\customer_churn\tuner0.json


In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

In [12]:
print("Starting Hyperparameter Search...")
tuner.search(
    X_train_scaled,y_train,
    epochs=20,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Trial 26 Complete [00h 00m 10s]
val_loss: 0.5752164125442505

Best val_loss So Far: 0.45924291014671326
Total elapsed time: 20h 20m 08s


what `tuner.search()` does behind the scenes:

1. **Pick:** The Tuner's "Oracle" selects a specific combination of hyperparameters to test.
2. **Build:** It passes those parameters to your `build_model` function to generate a fresh, untrained neural network in memory.
3. **Train:** It plugs your arguments (`epochs=20`, `validation_split`, `early_stop`) directly into a standard Keras `model.fit()` and trains the model.
4. **Evaluate & Save:** It checks the final Validation Loss, saves the score and weights to your hard drive, and destroys the model from memory.
5. **Repeat:** It uses the results to guess a *better* combination of parameters, loops back to Step 1, and repeats this process until it finds the absolute best model.

In short: It is just an automated `while` loop that continuously builds, trains, scores, and destroys models!

### The Oracle's Job

1. **The Blueprint Creator:** It looks at your limits (e.g., "choose between 32 and 128 neurons") and generates a specific blueprint (the `hp` object) for a new model to try.
2. **The Scorekeeper:** After the model finishes training, the Oracle looks at the final Validation Loss. It records this score in its internal database.
3. **The Strategist:** It uses that score to decide what to do next.

In [13]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Optimal Number of Layers: {best_hps.get('num_layers')}")
print(f"Optimal Learning Rate: {best_hps.get('learning_rate')}")

Optimal Number of Layers: 2
Optimal Learning Rate: 0.01


In [14]:
final_model = tuner.hypermodel.build(best_hps)
final_model.fit(
    X_train_scaled, y_train, 
    epochs=30, 
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30


C:\Users\Hp\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.7117 - loss: 0.5440 - val_accuracy: 0.7597 - val_loss: 0.5158
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7704 - loss: 0.4684 - val_accuracy: 0.7403 - val_loss: 0.5372
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7752 - loss: 0.4563 - val_accuracy: 0.7403 - val_loss: 0.5426
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7687 - loss: 0.4494 - val_accuracy: 0.7532 - val_loss: 0.5793


In [15]:
_, final_acc = final_model.evaluate(X_test_scaled, y_test)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7597 - loss: 0.5158

Final Test Accuracy: 0.7597


In [16]:
print(f"Baseline Accuracy (Guessed Parameters): {baseline_acc:.4f}")
print(f"Tuned Accuracy (Keras Tuner Optimized): {final_acc:.4f}")

Baseline Accuracy (Guessed Parameters): 0.7338
Tuned Accuracy (Keras Tuner Optimized): 0.7597


In [17]:
if final_acc > baseline_acc:
    print(f"Improvement: +{((final_acc - baseline_acc) / baseline_acc) * 100:.2f}%")
else:
    print("No improvement. The baseline guess was already highly optimal for this dataset split.")

Improvement: +3.54%
